# Dataset

## Retrieving data

In [1]:
from src.datasets import save_pages_to_jsonl

save_pages_to_jsonl(
    dataset_name="FineWeb Edu 2",
    num_pages=15,
    output_path="./data/fineweb_edu_2/fineweb_edu_2-train.jsonl",
    random_seed=163,
)
save_pages_to_jsonl(
    dataset_name="FineWeb Edu 2",
    num_pages=5,
    output_path="./data/fineweb_edu_2/fineweb_edu_2-validation.jsonl",
    random_seed=164,
)
save_pages_to_jsonl(
    dataset_name="FineWeb Edu 2",
    num_pages=5,
    output_path="./data/fineweb_edu_2/fineweb_edu_2-test.jsonl",
    random_seed=165,
)

## Uploading dataset to huggingface

In [ ]:
from huggingface_hub import HfApi, login
import os
from dotenv import load_dotenv

# Load token from .env file
load_dotenv()
token = os.getenv("HF_TOKEN")

# Login to Hugging Face
if token is None:
    login()  # This will prompt for token interactively
else:
    login(token=token)  # Login with token from .env

api = HfApi(token=token)

# Create the dataset repository
api.create_repo(
    repo_id="deepcoreCoalbiter/fineweb_edu_2", repo_type="dataset", private=False
)

# Upload the dataset files
api.upload_folder(
    repo_id="deepcoreCoalbiter/fineweb_edu_2",
    folder_path="./data/fineweb_edu_2",
    repo_type="dataset",
)

# Single Merge

In [1]:
import yaml

MODEL_NAME = "test3"
yaml_config = """
slices:
  - sources:
      - model: RoyJoy/llama-jan08
        layer_range: [0, 48]
      - model: RoyJoy/llama-jan16
        layer_range: [0, 48]
merge_method: slerp
base_model: RoyJoy/llama-jan08
parameters:
  t:
    - filter: self_attn
      value: [0, 0.5, 0.3, 0.7, 1]
    - filter: mlp
      value: [1, 0.5, 0.7, 0.3, 0]
    - value: 0.5
dtype: bfloat16
"""

# Save config as yaml file
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)

In [ ]:
# run merge
!mergekit-yaml config.yaml merge --lazy-unpickle

In [ ]:
!pip install -qU huggingface_hub
!pip install python-dotenv

from huggingface_hub import ModelCard, ModelCardData
from jinja2 import Template

username = "deepcoreCoalbiter"

template_text = """
# {{ model_name }}
"""

# Create a Jinja template object
jinja_template = Template(template_text.strip())

# Get list of models from config
data = yaml.safe_load(yaml_config)
if "models" in data:
    models = [data["models"][i]["model"] for i in range(len(data["models"])) if "parameters" in data["models"][i]]
elif "parameters" in data:
    models = [data["slices"][0]["sources"][i]["model"] for i in range(len(data["slices"][0]["sources"]))]
elif "slices" in data:
    models = [data["slices"][i]["sources"][0]["model"] for i in range(len(data["slices"]))]
else:
    raise Exception("No models or slices found in yaml config")

# Fill the template
content = jinja_template.render(
    model_name=MODEL_NAME,
)

# Save the model card
card = ModelCard(content)
card.save('merge/README.md')


In [ ]:
from huggingface_hub import HfApi

username = "deepcoreCoalbiter"

# Read token from .env file
import os
from dotenv import load_dotenv

load_dotenv()

api = HfApi(token=os.getenv("HF_TOKEN"))

api.create_repo(repo_id=f"{username}/{MODEL_NAME}", repo_type="model")
api.upload_folder(
    repo_id=f"{username}/{MODEL_NAME}",
    folder_path="merge",
)

# Evolutionary merge with custom task

In [1]:
yaml_config = """
genome:
    models:
      - tensoralchemistdev01/sv12
      - deepnet111/sn9-3b-star-009
    merge_method: slerp
    base_model: tensoralchemistdev01/sv12
    layer_granularity: 14

tasks:
  - name: fine_web_edu_2_ppl
    weight: 1.0
    metric: word_perplexity
"""

# Save config as yaml file
with open("evolve_config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)

In [2]:
!mergekit-evolve --strategy buffered --task-search-path lm-eval-tasks/fineweb-edu-2 --storage-path ./evolve_storage evolve_config.yaml

Resharding models:   0%|                                  | 0/2 [00:00<?, ?it/s]2025-01-31:07:25:44,066 INFO     [evolve.py:405] Using existing resharded model at ./evolve_storage/input_models/sv12_934583988
2025-01-31:07:25:44,066 INFO     [evolve.py:405] Using existing resharded model at ./evolve_storage/input_models/sn9-3b-star-009_3747678082
Resharding models: 100%|████████████████████████| 2/2 [00:00<00:00, 7619.08it/s]
2025-01-31:07:25:44,067 INFO     [evolve.py:405] Using existing resharded model at ./evolve_storage/input_models/sv12_934583988
2025-01-31 07:25:53,555	INFO worker.py:1841 -- Started a local Ray instance.
(4_w,8)-aCMA-ES (mu_w=2.6,w_1=52%) in dimension 4 (seed=401130, Fri Jan 31 07:25:55 2025)
Received 8 genotypes
(BufferedRayEvaluationStrategyActor pid=2461906) 2025-01-31:07:25:59,986 INFO     [strategy.py:151] Starting processing loop
(merge_model pid=2461912) 2025-01-31:07:26:04,533 INFO     [merge.py:81] Planning operations
(merge_model pid=2461912) 2025-01-31: